## **Refined**
***
Since this is a research project, the main thing here is the logic and approach to verifying/modifying parameters, not just the final metric.

In [1]:
# Import libs
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal

%matplotlib inline

In [2]:
# Define folders path
DATA_FOLDERS = [
    'Rupes A and B',
    'Guttural rupe',
    'Moan',
    'Grey Seal Data Additional'
]

OUTPUT_ROOT = 'outcome'
REFINED_OUTCOME_ROOT = 'outcome_refined'

In [3]:
# Create a folder for the source data, if it does not exist
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(REFINED_OUTCOME_ROOT, exist_ok=True)

In [4]:
# Spectrogram parameters
# Increase nfft and noverlap
fmin = 20
fmax = 2000  # from 1000 to 2000 to cover a wider range
nperseg = 4096  # was 2456
nfft = 8192  # was 4096
noverlap = 2048  # was 1228
window = 'hann'


# Number of "no-call" segments per audio file
NUM_NO_CALL_SEGMENTS = 5
# Duration (seconds) of each call and no-call segment
SEGMENT_DURATION = 1.0

## New Spectrogram Settings
- **nfft=8192**: doubled compared to the previous 4096 to achieve higher frequency resolution.
- **nperseg=4096**: the STFT window size has also been increased.
- **noverlap=2048**: increased overlap between windows, improving time resolution but increasing computational load.
- **fmax=2000 Hz**: previously, frequencies were limited to 1000 Hz; now 2000 Hz is used to capture higher harmonics, as seal vocalizations may contain higher frequencies.


In [5]:
# Debugging mode
DEBUG = True  # If True, prints additional information

In [6]:
# If WAV files are long and need splitting into shorter segments:
SPLIT_WAV = False  # True/False as needed
SEGMENT_SEC = 60  # e.g., 60 seconds per segment

# (If there are reasons for adding a time buffer, e.g., +0.2 s)
TIME_BUFFER = 0.0  # Adds (in seconds) to the start and end of the call

# For "no-call" segments
NUM_NO_CALL = 3  # Number of no-call segments to generate from one file
NO_CALL_LENGTH = 1.0  # Duration of 1 second (can be adjusted)

## 2. Processing Functions (split_wav, compute_spectrogram, slice_frequency, etc.)
Here have gathered functions that perform specific tasks:
- `ensure_mono`: if the audio is stereo, it takes the first channel.
- `split_wav`: splits audio into 60-second chunks (if needed).
- `compute_spectrogram`: calls `scipy.signal.spectrogram` with specified parameters.
- `slice_frequency`: slices the output spectrogram by the frequency range [fmin, fmax].
- `extract_call`: extracts the required segment by time and frequency.
- `pick_no_call_segments`: generates time windows without calls.

In [7]:
def split_wav(samples, sample_rate, segment_sec=60):
    """
    Splits audio into ~ segment_sec-long parts.
    Returns a list of numpy arrays.
    """
    total_samples = len(samples)
    segment_samples = int(segment_sec * sample_rate)
    
    segments = []
    start_idx = 0
    while start_idx < total_samples:
        end_idx = min(start_idx + segment_samples, total_samples)
        segment = samples[start_idx:end_idx]
        segments.append(segment)
        start_idx = end_idx
    return segments

### Why Split WAV Files?
- If the audio file is too large, computing the spectrogram for the entire file (with large `nfft/noverlap`) can be extremely time-consuming and require a lot of memory.
- Splitting the audio into smaller segments (e.g., 60 seconds) simplifies and speeds up the process.
- The `signal.spectrogram` is then applied to each segment, and annotated calls are searched within that segment (with the necessary time offset added).

In [8]:
def ensure_mono(samples):
    """
    Ensures that if the recording is stereo (2 channels), we select the first channel.
    """
    if len(samples.shape) > 1 and samples.shape[1] > 1:
        return samples[:, 0]
    return samples

In [9]:
def compute_spectrogram(samples, sample_rate):
    """
    Computes the spectrogram using scipy.signal.spectrogram
    with extended parameters (nfft, nperseg, noverlap).
    """
    freqs, times, Sxx = signal.spectrogram(
        samples,
        fs=sample_rate,
        nperseg=nperseg,
        nfft=nfft,
        noverlap=noverlap,
        window=window
    )
    # Remove very small values
    Sxx[Sxx < 0.001] = 0.001
    return freqs, times, Sxx

In [10]:
def slice_frequency(freqs, Sxx, fmin, fmax):
    """
    Slices the spectrogram by frequency range [fmin, fmax].
    Returns (freqs_new, Sxx_new).
    """
    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    if len(idx) == 0:
        return None, None
    freqs_new = freqs[idx]
    Sxx_new = Sxx[idx, :]
    return freqs_new, Sxx_new

In [11]:
def extract_call(freqs, times, Sxx, t_start, t_end, freq_range=(20, 2000)):
    """
    Extracts a spectrogram segment [t_start, t_end] and [freq_range].
    Returns (Sxx_sub, freqs_sub, times_sub) or (None, None, None).
    """
    time_idx = np.where((times >= t_start) & (times <= t_end))[0]
    freq_idx = np.where((freqs >= freq_range[0]) & (freq_range[1] >= freqs))[0]
    if len(time_idx) == 0 or len(freq_idx) == 0:
        return None, None, None
    
    Sxx_sub = Sxx[freq_idx][:, time_idx]
    freqs_sub = freqs[freq_idx]
    times_sub = times[time_idx]
    return Sxx_sub, freqs_sub, times_sub

In [12]:
def pad_spectrogram(Sxx_sub, max_fdim, max_tdim):
    """
    (If needed) Pads the spectrogram fragment Sxx_sub (freq x time) with zeros
    to (max_fdim x max_tdim).
    (In this example, padding is optional and depends on the need for a uniform size.)
    """
    fdim, tdim = Sxx_sub.shape
    pad_f = max_fdim - fdim
    pad_t = max_tdim - tdim
    if pad_f < 0 or pad_t < 0:
        # If larger than the "maximum" - trim it
        return Sxx_sub[:max_fdim, :max_tdim]
    # Otherwise, pad with zeros
    Sxx_padded = np.pad(
        Sxx_sub,
        ((0, pad_f), (0, pad_t)),
        mode='constant',
        constant_values=0
    )
    return Sxx_padded

In [13]:
def pick_no_call_segments(df_annot, total_duration, window_length=1.0, n_segments=3, seed=42):
    """
    Returns a list of (start_time, end_time) where no calls exist.
    Simple random selection.
    """
    import random
    random.seed(seed)
    
    call_intervals = []
    for _, row in df_annot.iterrows():
        c_start = row['Begin Time (s)']
        c_end = row['End Time (s)']
        call_intervals.append((c_start, c_end))
    call_intervals.sort(key=lambda x: x[0])
    
    no_call_windows = []
    start_all, end_all = 0, total_duration
    attempts = 0
    max_attempts = 1000

    while len(no_call_windows) < n_segments and attempts < max_attempts:
        attempts += 1
        rand_start = random.uniform(start_all, end_all - window_length)
        rand_end = rand_start + window_length

        # Check for overlap with calls
        overlap = False
        for (cs, ce) in call_intervals:
            if not (rand_end <= cs or rand_start >= ce):
                overlap = True
                break
        if not overlap:
            no_call_windows.append((rand_start, rand_end))
    
    return no_call_windows

## 3. Main File Processing Loop
In this block:
1. Iterate over all folders (`DATA_FOLDERS`).
2. Find `.wav` files in each folder.
3. For each `.wav`, read the audio and look for the corresponding `.txt` with annotations.
4. (Optionally) split the `.wav` into shorter segments.
5. Compute the spectrogram with the new parameters.
6. Extract "calls" (from annotations) and "no-calls."
7. Save the 2D arrays `.npy` in `outcome_refined/<folder>`.
8. Generate the new `metadata_refined.csv`.

In [14]:
all_refined_metadata = []  # where store the metadata

for data_folder in DATA_FOLDERS:
    folder_name = os.path.basename(data_folder)
    output_folder = os.path.join(REFINED_OUTCOME_ROOT, folder_name)
    os.makedirs(output_folder, exist_ok=True)

    # Search for .wav files
    wav_files = glob.glob(os.path.join(data_folder, '*.wav'))
    
    for wav_path in wav_files:
        base_name = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_folder, base_name + '.Table.1.selections.txt')

        if not os.path.exists(txt_path):
            print(f"[WARNING] No annotation file for {wav_path}")
            continue
        
        # Read audio
        sr, samples = wavfile.read(wav_path)
        samples = ensure_mono(samples)
        total_dur = len(samples) / sr

        # (Optional) Split WAV into segments
        if SPLIT_WAV:
            segments = split_wav(samples, sr, SEGMENT_SEC)
        else:
            segments = [samples]  # one segment = the entire file

        # Read annotations
        df_annot = pd.read_csv(txt_path, sep='\t')
        
        # If WAV is split, adjust annotation offsets
        segment_offset = 0.0  # Always 0 if SPLIT_WAV = False

        # Process each segment
        seg_index = 0
        for seg_samples in segments:
            # 1) Compute spectrogram
            freqs, times, Sxx = compute_spectrogram(seg_samples, sr)
            freqs, Sxx = slice_frequency(freqs, Sxx, fmin, fmax)
            if freqs is None or Sxx is None:
                seg_index += 1
                continue

            # 2) Extract calls
            for idx_row, row in df_annot.iterrows():
                t_start = row['Begin Time (s)'] - segment_offset
                t_end = row['End Time (s)'] - segment_offset

                # Add buffer if needed
                t_start = max(0, t_start - TIME_BUFFER)
                t_end = min(t_end + TIME_BUFFER, len(seg_samples) / sr)

                if t_start >= t_end:
                    continue

                annotation = row.get('Annotation', 'Unknown')
                Sxx_sub, f_sub, t_sub = extract_call(freqs, times, Sxx, t_start, t_end, (fmin, fmax))
                if Sxx_sub is None:
                    continue

                # Save .npy file
                output_filename = f"{base_name}_seg{seg_index}_call_{idx_row}.npy"
                np.save(os.path.join(output_folder, output_filename), Sxx_sub)

                meta = {
                    'source_wav': base_name,
                    'annotation_file': os.path.basename(txt_path),
                    'call_index': idx_row,
                    'call_type': annotation,
                    'begin_time': t_start + segment_offset,
                    'end_time': t_end + segment_offset,
                    'freq_min': fmin,
                    'freq_max': fmax,
                    'nfft': nfft,
                    'nperseg': nperseg,
                    'noverlap': noverlap,
                    'saved_spectrogram': output_filename,
                    'label': annotation
                }
                all_refined_metadata.append(meta)

            # 3) Extract no-call segments
            no_call_windows = pick_no_call_segments(
                df_annot,
                total_duration=(len(seg_samples) / sr),
                window_length=NO_CALL_LENGTH,
                n_segments=NUM_NO_CALL
            )
            for i, (nc_start, nc_end) in enumerate(no_call_windows):
                Sxx_sub, f_sub, t_sub = extract_call(freqs, times, Sxx, nc_start, nc_end, (fmin, fmax))
                if Sxx_sub is None:
                    continue

                output_filename = f"{base_name}_seg{seg_index}_nocall_{i}.npy"
                np.save(os.path.join(output_folder, output_filename), Sxx_sub)

                meta = {
                    'source_wav': base_name,
                    'annotation_file': os.path.basename(txt_path),
                    'call_index': i,
                    'call_type': 'no-call',
                    'begin_time': nc_start + segment_offset,
                    'end_time': nc_end + segment_offset,
                    'freq_min': fmin,
                    'freq_max': fmax,
                    'nfft': nfft,
                    'nperseg': nperseg,
                    'noverlap': noverlap,
                    'saved_spectrogram': output_filename,
                    'label': 'no-call'
                }
                all_refined_metadata.append(meta)

            seg_index += 1

In [15]:
# Save updated metadata
df_refined = pd.DataFrame(all_refined_metadata)
metadata_path = os.path.join(REFINED_OUTCOME_ROOT, 'metadata_refined.csv')
df_refined.to_csv(metadata_path, index=False)

In [16]:
print("Refined extraction complete!")
print(f"Saved spectrograms in: {REFINED_OUTCOME_ROOT}/...")
print(f"Metadata CSV: {metadata_path}")

Refined extraction complete!
Saved spectrograms in: outcome_refined/...
Metadata CSV: outcome_refined\metadata_refined.csv


## 4. Summary
As a result:
- An extended version of spectrograms (with higher frequency resolution).
- The ability to compare the effects of these settings in subsequent stages (e.g., in model training).

### Conclusion
The complete code consists of seven main parts:
1. Parameter declarations (`nfft`, `nperseg`, `noverlap`, `fmax`, etc.).
2. Processing functions (`split_wav`, `compute_spectrogram`, `slice_frequency`, `extract_call`, `pick_no_call_segments`).
3. Main loop for `.wav` file processing.
4. Saving results (spectrograms `*.npy` and `metadata_refined.csv`).
5. Markdown blocks with comments, explanations, and rationale for the changes.